# DLAI Model Merging - Kaggle smoke test

This notebook verifies the environment, downloads public benchmark data, trains very small specialist runs, and checks the merging functions. It is **not** a final experiment.

Kaggle settings: enable a GPU and enable Internet for the first run.

In [ ]:
!nvidia-smi
!python --version

## Fetch the project
The repository is public and contains no credentials. Re-running this cell starts from the latest committed code.

In [ ]:
import os
from pathlib import Path

REPO = 'https://github.com/LeuxLello/Dlai-model-merging.git'
WORKDIR = Path('/kaggle/working/Dlai-model-merging')
if not WORKDIR.exists():
    os.system(f'git clone {REPO} {WORKDIR}')
os.chdir(WORKDIR)
print(Path.cwd())

In [ ]:
%pip install -q -e .
import torch, transformers, datasets
print('torch', torch.__version__)
print('transformers', transformers.__version__)
print('datasets', datasets.__version__)
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Enable a GPU in Kaggle notebook settings.'

## Algorithm unit tests

In [ ]:
!pytest -q

## Tiny specialist runs
Only 256 training examples and 128 validation examples are used. The goal is to verify the full pipeline, not to obtain meaningful accuracy.

In [ ]:
from dlai_merge.training import TrainConfig, train_specialist

smoke_summaries = {}
for task in ['sst2', 'imdb', 'mrpc', 'rte']:
    print(f'\n===== {task.upper()} =====')
    config = TrainConfig(
        task=task,
        output_root='/kaggle/working/smoke_specialists',
        max_train_samples=256,
        max_eval_samples=128,
        epochs=1,
        train_batch_size=32,
        eval_batch_size=64,
    )
    smoke_summaries[task] = train_specialist(config)
smoke_summaries

## Minimal merge check
This loads two trained encoders and verifies that Mean, Task Arithmetic, and TIES return a complete compatible state.

In [ ]:
from pathlib import Path
import torch
from transformers import AutoModelForSequenceClassification
from dlai_merge.merging import mean_merge, task_arithmetic, ties_merge

base = AutoModelForSequenceClassification.from_pretrained('prajjwal1/bert-mini', num_labels=2)
base_state = {k: v.detach().cpu() for k, v in base.base_model.state_dict().items()}
root = Path('/kaggle/working/smoke_specialists')
sst2 = torch.load(root / 'sst2/seed-42/encoder.pt', map_location='cpu', weights_only=True)
imdb = torch.load(root / 'imdb/seed-42/encoder.pt', map_location='cpu', weights_only=True)
merged = {
    'mean': mean_merge(base_state, [sst2, imdb]),
    'task_arithmetic': task_arithmetic(base_state, [sst2, imdb], scale=0.5),
    'ties': ties_merge(base_state, [sst2, imdb], density=0.2, scale=1.0),
}
assert all(set(state) == set(base_state) for state in merged.values())
print('All three merged states are structurally valid.')

## Export compact smoke-test evidence

In [ ]:
import json
output = Path('/kaggle/working/smoke_summary.json')
output.write_text(json.dumps(smoke_summaries, indent=2), encoding='utf-8')
print(output)